In [8]:
import pandas as pd
import sys
sys.path.insert(0, '..')
from config import settings

report = pd.read_csv(settings.OUTPUT_DIR / 'validation_report.csv')
clean = pd.read_csv(settings.OUTPUT_DIR / 'valid_transactions.csv')
flagged = pd.read_csv(settings.OUTPUT_DIR / 'flagged_issues.csv')

In [9]:
# _explode
errors_series= flagged[['transaction_id', 'error_codes']].copy()
errors_exploded = errors_series.assign(
    error_code = errors_series['error_codes'].str.split(', ')
).explode('error_code')

errors_exploded[['transaction_id', 'error_code']].head(10)

,transaction_id,error_code
0,LC-0OXFQ8,LEI004
0,LC-0OXFQ8,DATE007
1,LC-Q2Y9V8,LEI004
1,LC-Q2Y9V8,SHIP002
2,LC-5LXO6Q,LEI004
3,LC-V55PSQ,DATE002
4,LC-DTJVZH,LEI004
4,LC-DTJVZH,XVAL002
5,LC-QIMQOD,LEI004
6,LC-UZFK8U,LEI004


In [10]:
# _lookup-table
from config.validation_rules import ALL_ERROR_CODES

error_lookup = pd.DataFrame([
    {'error_code': code, 'message': info['message'], 'severity': info['severity']}
    for code, info in ALL_ERROR_CODES.items()

])
print(f"Error lookup: {len(error_lookup)} codes")
error_lookup.head()

Error lookup: 54 codes


,error_code,message,severity
0,AMT001,Amount zero,CRITICAL
1,AMT002,Amount negative,CRITICAL
2,AMT003,Amount exceeds maximum allowed,HIGH
3,AMT004,Amount below minimum allowed,HIGH
4,AMT005,Invalid amount format,CRITICAL


In [11]:
# _merge -1
errors_enriched = errors_exploded.merge(error_lookup, on='error_code')
print(f"Exploded: {len(errors_exploded)} rows")
print(f"Enriched: {len(errors_enriched)} rows")
errors_enriched[['transaction_id', 'error_code', 'severity', 'message']].head(10)

Exploded: 237 rows
Enriched: 237 rows


,transaction_id,error_code,severity,message
0,LC-0OXFQ8,LEI004,HIGH,LEI not found in GLEIF.md database
1,LC-0OXFQ8,DATE007,MEDIUM,LC validity period exceeds maximum
2,LC-Q2Y9V8,LEI004,HIGH,LEI not found in GLEIF.md database
3,LC-Q2Y9V8,SHIP002,HIGH,Port of loading missing
4,LC-5LXO6Q,LEI004,HIGH,LEI not found in GLEIF.md database
5,LC-V55PSQ,DATE002,HIGH,Date invalid format
6,LC-DTJVZH,LEI004,HIGH,LEI not found in GLEIF.md database
7,LC-DTJVZH,XVAL002,MEDIUM,Issuing bank country does not match applicant ...
8,LC-QIMQOD,LEI004,HIGH,LEI not found in GLEIF.md database
9,LC-UZFK8U,LEI004,HIGH,LEI not found in GLEIF.md database


In [12]:
# _merge -2
errors_enriched = errors_exploded.merge(
    error_lookup,
    on='error_code',
    how='left',
    validate='many_to_one'
)
print(f"Rows: {len(errors_enriched)}")
errors_enriched[['transaction_id', 'error_code', 'severity', 'message']].head(10)

Rows: 237


,transaction_id,error_code,severity,message
0,LC-0OXFQ8,LEI004,HIGH,LEI not found in GLEIF.md database
1,LC-0OXFQ8,DATE007,MEDIUM,LC validity period exceeds maximum
2,LC-Q2Y9V8,LEI004,HIGH,LEI not found in GLEIF.md database
3,LC-Q2Y9V8,SHIP002,HIGH,Port of loading missing
4,LC-5LXO6Q,LEI004,HIGH,LEI not found in GLEIF.md database
5,LC-V55PSQ,DATE002,HIGH,Date invalid format
6,LC-DTJVZH,LEI004,HIGH,LEI not found in GLEIF.md database
7,LC-DTJVZH,XVAL002,MEDIUM,Issuing bank country does not match applicant ...
8,LC-QIMQOD,LEI004,HIGH,LEI not found in GLEIF.md database
9,LC-UZFK8U,LEI004,HIGH,LEI not found in GLEIF.md database


In [13]:
# _extract
error_categories = errors_enriched.str.extract(r'^([A-Z]+)')[0]


AttributeError: 'DataFrame' object has no attribute 'str'